# DWH Stock OHLCV Manual Smoke V0.1

> Manual smoke test only. This notebook writes to `_manual_smoke` path and **does not** update canonical DWH registry, SourceSpec, or normalized layers.

In [ ]:
# 1) clone / cd / install
# Run in a fresh Colab runtime.
!git clone https://github.com/<your-org>/<your-repo>.git
%cd /content/<your-repo>
!pip install -q -U pip
!pip install -q -e . pyarrow akshare

In [ ]:
# 2) optional branch checkout cell
# Uncomment and edit branch name if needed.
# !git fetch --all
# !git checkout <your-branch>
# !git pull

In [ ]:
# 3) inspect available stock OHLCV adapter signature
import inspect
from qsys.data.sources.akshare_market import fetch_stock_zh_a_hist

print(fetch_stock_zh_a_hist)
print(inspect.signature(fetch_stock_zh_a_hist))
print(inspect.getdoc(fetch_stock_zh_a_hist))

In [ ]:
# 4) fetch 3 symbols for 2024
from pathlib import Path
import pandas as pd

symbols = ["000001.SZ", "000002.SZ", "600000.SH"]
start_date = "2024-01-01"
end_date = "2024-12-31"
manual_smoke_root = Path("/content/drive/MyDrive/a_share_quant_cache/raw/_manual_smoke/stock_ohlcv/v1")
manual_smoke_root.mkdir(parents=True, exist_ok=True)

def to_ak_symbol(ts_code: str) -> str:
    code, exch = ts_code.split(".")
    return f"{code}{exch.lower()}"

fetched = {}
errors = {}
for ts_code in symbols:
    try:
        res = fetch_stock_zh_a_hist(
            symbol=to_ak_symbol(ts_code),
            start_date=start_date.replace("-", ""),
            end_date=end_date.replace("-", ""),
            period="daily",
            adjust="qfq",
        )
        df = res.raw.copy()
        fetched[ts_code] = df
        print(f"OK {ts_code}: rows={len(df)}")
    except Exception as e:
        errors[ts_code] = str(e)
        print(f"FAIL {ts_code}: {e}")

In [ ]:
# 5) write one parquet per symbol under manual smoke root
written_paths = []
for ts_code, df in fetched.items():
    out_dir = manual_smoke_root / f"symbol={ts_code}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_fp = out_dir / "data.parquet"
    df.to_parquet(out_fp, index=False)
    written_paths.append(out_fp)

print("written files:")
for p in written_paths:
    print(p)

In [ ]:
# 6) audit written files
inventory = []
for ts_code in symbols:
    fp = manual_smoke_root / f"symbol={ts_code}" / "data.parquet"
    exists = fp.exists()
    size = fp.stat().st_size if exists else 0
    inventory.append({"symbol": ts_code, "path": str(fp), "exists": exists, "size_bytes": size, "size_mb": size/1024/1024 if size else 0.0})

inv_df = pd.DataFrame(inventory)
inv_df

In [ ]:
# 7) display columns, date range, row counts, missing values
def pick_date_column(df: pd.DataFrame):
    for c in ["日期", "date", "trade_date", "时间"]:
        if c in df.columns:
            return c
    return None

audit_rows = []
for ts_code in symbols:
    fp = manual_smoke_root / f"symbol={ts_code}" / "data.parquet"
    if not fp.exists():
        audit_rows.append({"symbol": ts_code, "status": "missing_file"})
        continue
    df = pd.read_parquet(fp)
    date_col = pick_date_column(df)
    dt_min = dt_max = None
    if date_col is not None and not df.empty:
        dt = pd.to_datetime(df[date_col], errors="coerce")
        dt_min = str(dt.min().date()) if dt.notna().any() else None
        dt_max = str(dt.max().date()) if dt.notna().any() else None
    audit_rows.append({
        "symbol": ts_code,
        "status": "ok",
        "rows": len(df),
        "columns": list(df.columns),
        "date_col": date_col,
        "min_date": dt_min,
        "max_date": dt_max,
        "null_counts": df.isna().sum().to_dict(),
    })

audit_df = pd.DataFrame([{k: v for k, v in r.items() if k not in {"columns", "null_counts"}} for r in audit_rows])
audit_df

In [ ]:
# 8) write smoke_manifest.json and smoke_inventory.csv
import json
from datetime import datetime, timezone

manifest = {
    "note": "Manual smoke only; not canonical DWH raw dataset.",
    "source_adapter": "qsys.data.sources.akshare_market.fetch_stock_zh_a_hist",
    "symbols": symbols,
    "start_date": start_date,
    "end_date": end_date,
    "output_root": str(manual_smoke_root),
    "written_symbols": [s for s in symbols if (manual_smoke_root / f"symbol={s}" / "data.parquet").exists()],
    "failed_symbols": errors,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}

(manual_smoke_root / "smoke_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
inv_df.to_csv(manual_smoke_root / "smoke_inventory.csv", index=False)
print(manual_smoke_root / "smoke_manifest.json")
print(manual_smoke_root / "smoke_inventory.csv")

## 9) Important scope note

This notebook is a **manual smoke test only** for stock OHLCV raw fetching.

- It writes to: `/content/drive/MyDrive/a_share_quant_cache/raw/_manual_smoke/stock_ohlcv/v1/`
- It does **not** write to canonical `raw/stock_ohlcv/v1/`
- It does **not** add SourceSpec / RawWarehouseRunner integration
- It does **not** implement normalized price panel, factors, or backtests